# Heart — XAI extension

The reusable XAI framework is applied to the pre-specified L2 logistic-regression model with 10-fold cross-validation, using the stratified 50/50 split.

Standardisation is fitted on the training partition before internal cross-validation; it is not refitted within each fold.

The framework generates explanations and assesses retraining-based stability, using ROC-AUC for scoring. Results are exported to `../../results/heart`.

## Minimal data and model preparation

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV

file_path = "../../data/heart/processed.cleveland.data"

# column names -> defined manually
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak',
    'slope', 'ca', 'thal', 'target'
]

df = pd.read_csv(file_path, names=column_names)

# replace missing values '?'
df.replace('?', np.nan, inplace=True)

# conversion numeric columns
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# drop missing values
df = df.dropna()

# conversion to binary classification
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

X = df.drop('target', axis=1)
y = df['target']

feature_names = X.columns.tolist()

# The source paper uses equal training and test subsets. Split before
# learning preprocessing parameters to prevent test-data leakage.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.50,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=feature_names,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=feature_names,
    index=X_test_raw.index,
)

In [2]:
lr_10fold = LogisticRegressionCV(Cs=10, cv=10, penalty='l2', solver='liblinear', max_iter=1000, random_state=42)
lr_10fold.fit(X_train, y_train)


LogisticRegressionCV(cv=10, max_iter=1000, random_state=42, solver='liblinear')

In [3]:
# Pre-specified XAI model: the 10-fold L2 logistic-regression variant.
# It is chosen from the replicated study design, not by inspecting the
# held-out test ranking above.
final_model = lr_10fold
selected_model_name = "Logistic Regression (10-fold CV)"

In [4]:
best_model = final_model

print("Final XAI model:", type(best_model).__name__)
print("Pre-specified model:", selected_model_name)
print("Selection did not use held-out test performance.")

Final XAI model: LogisticRegressionCV
Pre-specified model: Logistic Regression (10-fold CV)
Selection did not use held-out test performance.


In [5]:
from sklearn.base import clone

def train_model_fn(seed):

    model = clone(best_model)

    if hasattr(model, "random_state"):
        model.set_params(random_state=seed)

    model.fit(X_train, y_train)

    return model

## Call the existing reusable XAI framework

In [6]:
import sys
from pathlib import Path

# Notebook is in thesis/src/replication
# Framework is in thesis/src/xai_framework
FRAMEWORK_DIR = Path.cwd().parent / "xai_framework"

if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline

c:\Users\catia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parents[1]))

from run_pipeline import run_pipeline

config = {
    "dataset_name": "heart",
    "model": best_model,

    "X_train": X_train,
    "X_test": X_test,

    "y_train": y_train,
    "y_test": y_test,

    "feature_names": feature_names,

    "train_model_fn": train_model_fn,

    "scoring": "roc_auc",

    "output_dir": "../../results/heart"
}

experiment = run_pipeline(config)

PermutationExplainer explainer: 150it [00:44,  3.32it/s]                         


Results exported to: ../../results/heart


## XAI framework output sanity checks

In [8]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))

Dataset: heart
Model: LogisticRegressionCV
X_train shape: (148, 13)
X_test shape: (149, 13)
y_train length: 148
y_test length: 149
Feature names: 13


#### Consistency

In [9]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("Basic shape validation: OK")

Basic shape validation: OK


#### Fitted model predicts on the configured data

In [10]:
model = config["model"]
X_test = config["X_test"]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)[:, 1]
    print("Probability range:", probs.min(), probs.max())

preds = model.predict(X_test)

print("Predictions:", np.unique(preds, return_counts=True))
print("Prediction test: OK")

Probability range: 0.26801784968312586 0.7620684993860655
Predictions: (array([0, 1]), array([80, 69]))
Prediction test: OK


#### Results from Framework

In [11]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print(global_results.keys())
print(local_results.keys())

if stability_results is not None:
    print(stability_results.keys())
else:
    print("Stability: not available")

dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'rank_compare', 'rank_agreement'])
dict_keys(['cases', 'lime'])
dict_keys(['raw', 'summary', 'rank_stability'])


#### Global XAI

In [12]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("Native:")
display(native_df.head())

print("Permutation:")
display(perm_df.head())

print("SHAP:")
display(shap_df.head())

Native:


,Feature,Importance,RawValue,Method
12,thal,0.153951,0.153951,coefficients
2,cp,0.130170,0.130170,coefficients
8,exang,0.114381,0.114381,coefficients
7,thalach,0.104741,-0.104741,coefficients
11,ca,0.100806,0.100806,coefficients


Permutation:


,Feature,Importance,Std
12,thal,0.035833,0.013175
11,ca,0.029493,0.008113
1,sex,0.014293,0.005012
7,thalach,0.011286,0.006377
2,cp,0.010707,0.008875


SHAP:


,Feature,MeanAbsSHAP,MeanSHAP
0,thal,0.036146,-0.002521
1,cp,0.026454,-0.001569
2,exang,0.025674,-0.001041
3,thalach,0.024346,0.005803
4,slope,0.022205,0.000467


#### Feature coverage

In [13]:
if native_df is not None:
    print("Native features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", len(config["feature_names"]))

Native features: 13
Permutation features: 13
SHAP features: 13
Expected features: 13


#### Broken values

In [14]:
print("Permutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())

Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0


#### Rank agreement

In [15]:
print(global_results["rank_agreement"])

display(global_results["rank_compare"].head())

{'Native_vs_SHAP': np.float64(0.9780219780219779), 'Native_vs_Permutation': np.float64(0.6208791208791209), 'SHAP_vs_Permutation': np.float64(0.6153846153846153)}


,Feature,Importance_x,RawValue,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,thal,0.153951,0.153951,coefficients,1.0,0.036146,-0.002521,1.0,0.035833,0.013175,1.0
1,cp,0.130170,0.130170,coefficients,2.0,0.026454,-0.001569,2.0,0.010707,0.008875,5.0
2,exang,0.114381,0.114381,coefficients,3.0,0.025674,-0.001041,3.0,0.005888,0.007579,7.0
3,thalach,0.104741,-0.104741,coefficients,4.0,0.024346,0.005803,4.0,0.011286,0.006377,4.0
4,ca,0.100806,0.100806,coefficients,5.0,0.022152,0.001595,6.0,0.029493,0.008113,2.0


#### Representative cases

In [16]:
cases = local_results["cases"]

display(
    cases[
        ["CaseType", "y_true", "y_pred", "y_prob"]
    ]
)

,CaseType,y_true,y_pred,y_prob
84,TP,1,1,0.762068
47,TN,0,0,0.268018
95,FP,0,1,0.677475
4,FN,1,0,0.364029
44,Borderline,0,1,0.500092


#### LIME

In [17]:
lime_df = local_results["lime"]

print("LIME rows:", len(lime_df))
display(lime_df.head())

LIME rows: 50


,CaseIndex,Prediction,Probability,Rule,Weight
0,84,1,0.762068,0.32 < cp <= 0.85,0.059946
1,84,1,0.762068,thalach <= -0.52,0.047814
2,84,1,0.762068,ca > 0.40,0.039839
3,84,1,0.762068,-0.70 < exang <= 1.42,0.038325
4,84,1,0.762068,-0.94 < thal <= 1.11,0.036612


#### Stability

In [18]:
if stability_results is not None:
    display(stability_results["raw"].head())
    print(stability_results["summary"])

,Feature,Importance,ImportanceStd,Seed,Rank
0,age,0.000978,0.002386,1,10.0
1,sex,0.014094,0.006878,1,3.0
2,cp,0.006359,0.007545,1,7.0
3,trestbps,0.000779,0.002603,1,11.0
4,chol,0.000254,0.000390,1,12.0


     Feature  MeanImportance  StdImportance  MeanRank   StdRank
10      thal        0.032243       0.001692       1.0  0.000000
1         ca        0.030478       0.001933       2.0  0.000000
8        sex        0.013007       0.001564       3.2  0.447214
11   thalach        0.011446       0.002081       4.2  1.095445
4      exang        0.009880       0.001006       5.2  0.836660
3         cp        0.008844       0.001979       6.0  1.000000
6    oldpeak        0.008170       0.001531       6.4  0.894427
9      slope        0.001931       0.000974       8.6  0.894427
12  trestbps        0.001391       0.000677       9.0  1.224745
0        age        0.000692       0.000535      10.4  1.140175
2       chol        0.000268       0.000095      11.0  0.707107
7    restecg        0.000152       0.001213      11.0  1.414214
5        fbs       -0.001793       0.000590      13.0  0.000000


#### Exported folder

In [19]:
from pathlib import Path

out_dir = Path(config["output_dir"]).resolve()

print("Output directory:", out_dir)

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))

Output directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\heart
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
lime_explanations.csv
model.pkl
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv
stability.csv
stability_rank.csv
stability_summary.csv


In [20]:
print(
    experiment.state.results["local"].keys()
)

dict_keys(['cases', 'lime'])


In [21]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)

['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
84,-0.411115,0.619013,0.847174,0.506570,0.853934,-0.428952,-0.957016,-1.484606,1.421411,2.757194,0.703526,2.659046,1.114241,1,1,0.762068,0.262068,TP,84
47,-0.962248,-1.615474,-1.260082,-1.567766,-0.798811,-0.428952,-0.957016,0.932043,-0.703526,-0.915126,-0.949202,-0.733530,-0.940683,0,0,0.268018,0.231982,TN,47
95,1.021829,0.619013,0.847174,-0.204630,0.238550,-0.428952,-0.957016,-2.306267,1.421411,-0.740254,0.703526,0.397329,1.114241,0,1,0.677475,0.177475,FP,95
4,0.360470,-1.615474,-1.260082,0.269504,1.223165,2.331262,1.052039,-0.034617,-0.703526,-0.915126,-0.949202,1.528188,-0.940683,1,0,0.364029,0.135971,FN,4
44,-0.080436,0.619013,-0.206454,1.099238,-0.306504,-0.428952,1.052039,0.593712,-0.703526,0.483853,-0.949202,-0.733530,1.114241,0,1,0.500092,0.000092,Borderline,44


In [22]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns

print("CaseIndex exists in memory.")

CaseIndex exists in memory.


In [23]:
heart_cases = pd.read_csv(
    Path("../../results/heart/representative_cases.csv")
)

print(heart_cases.columns.tolist())
display(heart_cases)

['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
0,-0.411115,0.619013,0.847174,0.506570,0.853934,-0.428952,-0.957016,-1.484606,1.421411,2.757194,0.703526,2.659046,1.114241,1,1,0.762068,0.262068,TP,84
1,-0.962248,-1.615474,-1.260082,-1.567766,-0.798811,-0.428952,-0.957016,0.932043,-0.703526,-0.915126,-0.949202,-0.733530,-0.940683,0,0,0.268018,0.231982,TN,47
2,1.021829,0.619013,0.847174,-0.204630,0.238550,-0.428952,-0.957016,-2.306267,1.421411,-0.740254,0.703526,0.397329,1.114241,0,1,0.677475,0.177475,FP,95
3,0.360470,-1.615474,-1.260082,0.269504,1.223165,2.331262,1.052039,-0.034617,-0.703526,-0.915126,-0.949202,1.528188,-0.940683,1,0,0.364029,0.135971,FN,4
4,-0.080436,0.619013,-0.206454,1.099238,-0.306504,-0.428952,1.052039,0.593712,-0.703526,0.483853,-0.949202,-0.733530,1.114241,0,1,0.500092,0.000092,Borderline,44


In [24]:
import sys
import sklearn
import numpy
import pandas
import shap
import statsmodels

print("Python:", sys.executable)
print("Python version:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("SHAP:", shap.__version__)
print("statsmodels:", statsmodels.__version__)

Python: c:\Users\catia\AppData\Local\Programs\Python\Python312\python.exe
Python version: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
scikit-learn: 1.6.1
NumPy: 2.1.3
pandas: 2.2.3
SHAP: 0.51.0
statsmodels: 0.14.6
